In [14]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import random
from itertools import combinations
from scipy.spatial import Delaunay


# ================= PARAMETERS =================

# ER done, AS done, APOL done  

graph_type = "RCG"

sizes = [80, 120, 160, 200, 300]

budget_frac = 0.15
target_save_fraction = 0.30   # fraction of remaining vertices to save
p_fire = 1.0
l_ci = 2
quantum_steps = 20
mc_runs = 40
seed = 42

# --- PSO / ACO parameters ---

pso_particles = 40
pso_iters = 60

aco_ants = 35
aco_iters = 60

alpha = 1.0
beta = 2.0
rho = 0.1

simulation_runs = 25

random.seed(seed)
np.random.seed(seed)


def kleinberg_small_world_graph(n, p=2, q=1, r=2, seed=None):

    if seed is not None:
        random.seed(seed)

    G = nx.grid_2d_graph(n, n, periodic=False)

    # convert nodes to integers
    G = nx.convert_node_labels_to_integers(G)

    nodes = list(G.nodes())

    # Manhattan distance
    def dist(u,v):
        x1,y1 = divmod(u,n)
        x2,y2 = divmod(v,n)
        return abs(x1-x2)+abs(y1-y2)

    for u in nodes:

        probs = []
        others = []

        for v in nodes:
            if u != v:
                d = dist(u,v)
                probs.append(d**(-r))
                others.append(v)

        probs = np.array(probs)
        probs = probs / probs.sum()

        targets = np.random.choice(others, q, replace=False, p=probs)

        for v in targets:
            G.add_edge(u,v)

    return G




def random_apollonian_graph(n, seed=None):

    if seed is not None:
        random.seed(seed)

    G = nx.complete_graph(3)
    faces = [(0,1,2)]
    next_node = 3

    while next_node < n:

        f = random.choice(faces)
        a,b,c = f

        G.add_node(next_node)
        G.add_edge(next_node,a)
        G.add_edge(next_node,b)
        G.add_edge(next_node,c)

        faces.remove(f)
        faces.append((a,b,next_node))
        faces.append((a,c,next_node))
        faces.append((b,c,next_node))

        next_node += 1

    return G


# ================= k-EDGE CONNECTED SUM =================

def k_edge_connected_sum(G1, G2, k=3, seed=None):
    """
    Connect two graphs by adding k edges between random vertices of each graph.
    """
    if seed is not None:
        random.seed(seed)

    # relabel G2 nodes so labels do not collide
    offset = max(G1.nodes()) + 1
    G2_shift = nx.relabel_nodes(G2, lambda x: x + offset)

    G = nx.compose(G1, G2_shift)

    nodes1 = list(G1.nodes())
    nodes2 = list(G2_shift.nodes())

    for _ in range(k):
        u = random.choice(nodes1)
        v = random.choice(nodes2)
        G.add_edge(u, v)

    return G


# ================= GRAPH-OF-GRAPHS CONSTRUCTION =================

def graph_of_graphs(meta_graph, base_graph_generator, size=20):
    """
    Replace each node of a meta graph by a full graph.
    """

    G = nx.Graph()
    mapping = {}
    offset = 0

    for node in meta_graph.nodes():
        H = base_graph_generator(size)

        H = nx.relabel_nodes(H, lambda x: x + offset)

        mapping[node] = list(H.nodes())
        G = nx.compose(G, H)

        offset = max(G.nodes()) + 1

    # connect subgraphs according to meta graph edges
    for u, v in meta_graph.edges():
        a = random.choice(mapping[u])
        b = random.choice(mapping[v])
        G.add_edge(a, b)

    return G


# ================= MANIFOLD CONNECTED SUM ANALOGY =================

def manifold_connected_sum(G1, G2, boundary_size=3, seed=None):
    """
    Remove small boundary neighborhoods and connect them.
    """
    if seed is not None:
        random.seed(seed)

    offset = max(G1.nodes()) + 1
    G2_shift = nx.relabel_nodes(G2, lambda x: x + offset)

    G = nx.compose(G1, G2_shift)

    nodes1 = random.sample(list(G1.nodes()), boundary_size)
    nodes2 = random.sample(list(G2_shift.nodes()), boundary_size)

    for u, v in zip(nodes1, nodes2):
        G.add_edge(u, v)

    return G




    
# ================= GRAPH GENERATOR =================
# (UNCHANGED — your full generator remains here)
def make_graph(n, gtype):
    if gtype == "ER":
        return nx.erdos_renyi_graph(n, 0.01, seed=seed)

    elif gtype == "BA":
        return nx.barabasi_albert_graph(n, 3, seed=seed)

    elif gtype == "WS":
        return nx.watts_strogatz_graph(n, 6, 0.01, seed=seed)

    #elif gtype == "RR":
        #return nx.random_regular_graph(3, n, seed=seed)

    elif gtype == "BB":
        m = 3
        G = nx.complete_graph(m)
        fitness = np.random.uniform(0.5, 1.5, size=n)
        for i in range(m, n):
            G.add_node(i)
            deg = np.array([G.degree(v) for v in G.nodes()])
            fit = fitness[:len(G.nodes())]
            prob = deg * fit
            prob /= prob.sum()
            targets = np.random.choice(list(G.nodes()), m,
                                       replace=False, p=prob)
            for t in targets:
                G.add_edge(i, t)
        return G

    elif gtype == "PLC":
        return nx.powerlaw_cluster_graph(n, 3, 0.2, seed=seed)

    elif gtype == "RG":
        return nx.random_geometric_graph(n, radius=0.2, seed=seed)

    elif gtype == "SBM":
        sizes_local = [n//3, n//3, n - 2*(n//3)]
        probs = [[0.1,0.02,0.02],
                 [0.02,0.1,0.02],
                 [0.02,0.02,0.1]]
        return nx.stochastic_block_model(sizes_local, probs, seed=seed)

    elif gtype == "KRON":

    # initiator probabilities
        a, b, c, d = 0.9, 0.5, 0.5, 0.1

        k = int(np.ceil(np.log2(n)))
        N = 2 ** k

    # expected number of edges
        m = int(4 * n)   # you can tune this

        G = nx.Graph()

        for _ in range(m):

            u = 0
            v = 0

            step = N // 2

            for _ in range(k):

                r = random.random()

                if r < a:
                    pass
                elif r < a + b:
                    v += step
                elif r < a + b + c:
                    u += step
                else:
                    u += step
                    v += step

                step //= 2

            if u != v:
                G.add_edge(u, v)

    # trim to n nodes
        if N > n:
            G = G.subgraph(range(n)).copy()

    # ensure connectivity
        if not nx.is_connected(G):

            components = list(nx.connected_components(G))

            for i in range(len(components) - 1):
                u = random.choice(list(components[i]))
                v = random.choice(list(components[i + 1]))
                G.add_edge(u, v)

        return G
    

    # ===== HARD GRAPH CLASSES =====

    elif gtype == "RINGC":
        k = max(4, n//5)
        num = n // k
        G = nx.Graph()
        start = 0
        reps = []

        for _ in range(num):
            H = nx.complete_graph(k)
            H = nx.relabel_nodes(H, lambda x: x + start)
            G = nx.compose(G, H)
            reps.append(start)
            start += k

        for i in range(num):
            G.add_edge(reps[i], reps[(i+1) % num])

        return G

    elif gtype == "SBM_HARD":
        sizes_local = [n//2, n - n//2]
        probs = [[0.15, 0.001],
                 [0.001, 0.15]]
        return nx.stochastic_block_model(sizes_local, probs, seed=seed)

    elif gtype == "CHAIN":
        k = max(3, n//6)
        G = nx.complete_graph(k)
        last = k - 1
        remaining = n - k

        while remaining > 0:
            size = min(k, remaining)
            H = nx.complete_graph(size)
            H = nx.relabel_nodes(H, lambda x: x + G.number_of_nodes())
            G = nx.compose(G, H)

            G.add_edge(last, G.number_of_nodes() - size)

            last = G.number_of_nodes() - 1
            remaining -= size

        return G

    elif gtype == "BBL":
        m1 = max(3, n//4)
        m2 = max(1, n - 2*m1)
        return nx.barbell_graph(m1, m2)

    elif gtype == "DB":
        k = max(3, n//3)
        path_len = max(1, n - 2*k)

        G1 = nx.complete_graph(k)
        G2 = nx.complete_graph(k)
        G2 = nx.relabel_nodes(G2, lambda x: x + k)
        G = nx.compose(G1, G2)

        last = k - 1
        for i in range(path_len):
            G.add_edge(last + i, last + i + 1)

        G.add_edge(last + path_len, k)
        return G

    elif gtype == "LOP":
        m = max(3, n//3)
        p = n - m
        return nx.lollipop_graph(m, p)

    elif gtype == "GRID":
        m = int(np.sqrt(n))
        G = nx.grid_2d_graph(m, m)
        return nx.convert_node_labels_to_integers(G)

    elif gtype == "TREE":
        return nx.random_labeled_tree(n, seed=seed)

    elif gtype == "2C":
        k = n // 2
        G1 = nx.complete_graph(k)
        G2 = nx.complete_graph(n - k)
        G2 = nx.relabel_nodes(G2, lambda x: x + k)
        G = nx.compose(G1, G2)
        G.add_edge(0, k)
        return G

    elif gtype == "SBM_STRONG":
        sizes_local = [n//2, n - n//2]
        probs = [[0.12,0.005],
                 [0.005,0.12]]
        return nx.stochastic_block_model(sizes_local, probs, seed=seed)

    elif gtype == "CP":
        core = max(5, n // 5)
        G = nx.complete_graph(core)

        # Add peripheral nodes
        for i in range(core, n):
            G.add_node(i)

        # Connect periphery node to core with high probability
        for j in range(core):
            if random.random() < 0.7:
                G.add_edge(i, j)

        # Optional: sparse periphery-periphery links
        for j in range(core, i):
            if random.random() < 0.05:
                G.add_edge(i, j)

        for i in range(core, n):
            G.add_node(i)
            targets = np.random.choice(range(core),
                                       size=min(2, core),
                                       replace=False)
            for t in targets:
                G.add_edge(i, t)
        return G

    elif gtype == "TORUS":  
        m = int(np.sqrt(n))
        if m*m != n:
            m = int(np.sqrt(n))
            n = m*m  # enforce square size

        G = nx.grid_2d_graph(m, m, periodic=True)
        return nx.convert_node_labels_to_integers(G) 


    elif gtype == "TORUS_TRI":
         m = int(np.sqrt(n))
         if m*m != n:
             m = int(np.sqrt(n))
             n = m*m

         G = nx.triangular_lattice_graph(m, m, periodic=True)
         return nx.convert_node_labels_to_integers(G)  



    elif gtype == "TORUS_IRREG":

         pts = np.random.rand(n, 2)   # torus coordinates in [0,1]^2
         G = nx.Graph()

         for i in range(n):
             G.add_node(i, pos=pts[i])

         for i in range(n):
             for j in range(i+1, n):

                 dx = abs(pts[i,0] - pts[j,0])
                 dy = abs(pts[i,1] - pts[j,1])

                 # periodic boundary (torus metric)
                 dx = min(dx, 1-dx)
                 dy = min(dy, 1-dy)

                 dist = np.sqrt(dx*dx + dy*dy)

                 # random radius -> irregular structure
                 r = np.random.uniform(0.08, 0.22)

                 if dist < r:
                     G.add_edge(i,j)

         return G  


    elif gtype == "TORUS_DELAUNAY":

         from scipy.spatial import Delaunay

         pts = np.random.rand(n,2)

         # replicate points to enforce toroidal periodicity
         shifts = np.array([[0,0],[1,0],[-1,0],[0,1],[0,-1],[1,1],[-1,-1]])
         tiled = np.vstack([pts + s for s in shifts])

         tri = Delaunay(tiled)

         G = nx.Graph()
         for i in range(n):
             G.add_node(i, pos=pts[i])

         for simplex in tri.simplices:
             for a,b in combinations(simplex,2):

                 a = a % n
                 b = b % n

                 if a != b:
                     G.add_edge(a,b)

         # randomly remove edges → strong irregularity
         for u,v in list(G.edges()):
             if np.random.rand() < 0.3:
                 G.remove_edge(u,v)

         return G 

    elif gtype == "TORUS_SCALEFREE":

         pts = np.random.rand(n,2)

         G = nx.Graph()

         for i in range(n):
             G.add_node(i, pos=pts[i])

         m0 = 5
         m = 3

         # initial clique
         for i in range(m0):
             for j in range(i+1,m0):
                 G.add_edge(i,j)

         for new in range(m0,n):

             weights = []
             targets = list(range(new))

             for v in targets:

                 dx = abs(pts[new,0] - pts[v,0])
                 dy = abs(pts[new,1] - pts[v,1])

                 # torus metric
                 dx = min(dx,1-dx)
                 dy = min(dy,1-dy)

                 dist = np.sqrt(dx*dx + dy*dy)

                 degree = G.degree(v) + 1

                 # spatial preferential attachment
                 w = degree * np.exp(-5*dist)

                 weights.append(w)

             weights = np.array(weights)
             weights = weights / weights.sum()

             chosen = np.random.choice(targets, size=m, replace=False, p=weights)

             for v in chosen:
                 G.add_edge(new,v)

         return G


    elif gtype == "CL":
         deg = np.random.zipf(2.5, n)
         deg = np.maximum(deg, 1)
         return nx.expected_degree_graph(deg, seed=seed)

    elif gtype == "CONFIG":
         deg = np.random.zipf(2.5, n)
         deg = [max(1, int(d)) for d in deg]
         if sum(deg) % 2 == 1:
             deg[0] += 1
         G = nx.configuration_model(deg, seed=seed)
         return nx.Graph(G)    

    elif gtype == "RREG":
         d = max(2, int(np.log(n)))
         if d >= n:
             d = n-1
         return nx.random_regular_graph(d, n, seed=seed) 

    elif gtype == "NW":
         return nx.newman_watts_strogatz_graph(n, 6, 0.2, seed=seed) 

    elif gtype == "KLEIN":

         m = int(np.sqrt(n))
         G = kleinberg_small_world_graph(m)

         if len(G) > n:
             G = G.subgraph(list(G.nodes())[:n]).copy()

         return G

    elif gtype == "KNN":
         pts = np.random.rand(n,2)
         G = nx.Graph()
         for i in range(n):
             dist = np.linalg.norm(pts - pts[i], axis=1)
             idx = np.argsort(dist)[1:6]
             for j in idx:
                 G.add_edge(i,j)
         return G     

    elif gtype == "WAX":
         return nx.waxman_graph(n, alpha=0.4, beta=0.1, seed=seed)   

    elif gtype == "LOB":
         return nx.random_lobster(n, 0.5, 0.5, seed=seed)

    elif gtype == "PLT":
         return nx.random_powerlaw_tree(n, tries=5000, seed=seed)

    elif gtype == "KOUT":
         k = 3
         return nx.random_k_out_graph(n, k, 0.5, seed=seed)


    elif gtype == "SHELL":
         constructor = [(n//3, n//6, 0.5),
                        (n//3, n//6, 0.4),
                        (n - 2*(n//3), n//8, 0.3)]
         return nx.random_shell_graph(constructor, seed=seed)

    elif gtype == "PART":
         sizes_local = [n//4]*4
         return nx.random_partition_graph(sizes_local, 0.2, 0.02, seed=seed)  

    elif gtype == "GRP":
         return nx.gaussian_random_partition_graph(n, 10, 10, 0.3, 0.05, seed=seed)    

    elif gtype == "AS":
         return nx.random_internet_as_graph(n, seed=seed)

    elif gtype == "APOL":
         return random_apollonian_graph(n, seed)
    
    elif gtype == "DD":
         return nx.duplication_divergence_graph(n, 0.5, seed=seed)

    elif gtype == "PLSEQ":
         seq = nx.utils.powerlaw_sequence(n, 2.5)
         deg = [max(1,int(x)) for x in seq]
         if sum(deg)%2==1:
             deg[0]+=1
         G = nx.configuration_model(deg)
         return nx.Graph(G)
        
    elif gtype == "GENUS_G":

        g = 10

        m = int(np.sqrt(n//g))

        blocks = []

        for i in range(g):

            T = nx.grid_2d_graph(m,m,periodic=True)

            T = nx.convert_node_labels_to_integers(T,first_label=i*m*m)

            blocks.append(T)

        G = nx.compose_all(blocks)

        # connect blocks cyclically
        for i in range(g):

            a = i*m*m
            b = ((i+1)%g)*m*m

            G.add_edge(a,b)

        return G

    elif gtype == "CART":

         n1 = int(np.sqrt(n))
         n2 = int(np.sqrt(n))

         G1 = make_graph(n1, "BA")
         G2 = make_graph(n2, "ws")

         G = nx.cartesian_product(G1, G2)

         return nx.convert_node_labels_to_integers(G)
        


    elif gtype == "STRONG":
         n1 = int(np.sqrt(n))
         n2 = int(np.sqrt(n))

         G1 = make_graph(n1, "BA")
         G2 = make_graph(n2, "TREE")

         G = nx.strong_product(G1, G2)
         return nx.convert_node_labels_to_integers(G)  

    elif gtype == "TENSOR":
         n1 = int(np.sqrt(n))
         n2 = int(np.sqrt(n))

         G1 = make_graph(n1, "WS")
         G2 = make_graph(n2, "TORUS")

         G = nx.tensor_product(G1, G2)
         return nx.convert_node_labels_to_integers(G) 

    elif gtype == "CORONA":
         base = int(n / 4)

         G1 = make_graph(base, "BA")
         G2 = make_graph(4, "ER")

         G = nx.corona_product(G1, G2)

         if G.number_of_nodes() > n:
             G = G.subgraph(list(G.nodes())[:n]).copy()

         return nx.convert_node_labels_to_integers(G)    


    elif gtype == "CSUM":

         n1 = n // 2
         n2 = n - n1

         G1 = make_graph(n1, "PLC")
         G2 = make_graph(n2, "WS")

         G2 = nx.relabel_nodes(G2, lambda x: x + n1)

         G = nx.compose(G1, G2)

         # identify two vertices
         v1 = 0
         v2 = n1

         # connect them (connected sum style)
         G.add_edge(v1, v2)

         return G    

    elif gtype == "MIX":

         n1 = int(np.sqrt(n))
         n2 = int(np.sqrt(n))

         G1 = make_graph(n1, "TREE")
         G2 = make_graph(n2, "BA")

         G = nx.cartesian_product(G1, G2)

         # add random shortcuts
         nodes = list(G.nodes())
         for _ in range(n):
             u = random.choice(nodes)
             v = random.choice(nodes)
             if u != v:
                 G.add_edge(u, v)

         return nx.convert_node_labels_to_integers(G)


    elif gtype == "KEDGE":

        G1 = nx.barabasi_albert_graph(n//2, 3)
        G2 = nx.watts_strogatz_graph(n//2, 4, 0.2)

        G = k_edge_connected_sum(G1, G2, k=4)
        return G

    elif gtype == "GOG":

        meta = nx.watts_strogatz_graph(6, 2, 0.3)

        def base(size):
            return nx.barabasi_albert_graph(size, 2)

        G = graph_of_graphs(meta, base, size=n//6)
        return G

    elif gtype == "MANI":

        G1 = nx.barabasi_albert_graph(n//2, 2)
        G2 = nx.watts_strogatz_graph(n//2, 4, 0.3)

        G = manifold_connected_sum(G1, G2, boundary_size=4)
        return G


    elif gtype == "RCG":

        l = max(2, int(np.sqrt(n)))     # number of communities
        k = max(2, n // l)              # nodes per community
        p = 0.01

        G = nx.relaxed_caveman_graph(l, k, p, seed=seed)

        # trim or pad
        G = nx.convert_node_labels_to_integers(G)
        if G.number_of_nodes() > n:
            G = G.subgraph(range(n)).copy()

        if not nx.is_connected(G):
            largest_cc = max(nx.connected_components(G), key=len)
            G = G.subgraph(largest_cc).copy()

            # re-expand if needed
            while G.number_of_nodes() < n:
                u = random.choice(list(G.nodes()))
                new = G.number_of_nodes()
                G.add_node(new)
                G.add_edge(new, u)    

        return G  

    elif gtype == "KARATE":

        base = nx.karate_club_graph()

        # replicate graph to reach n
        G = nx.disjoint_union_all([base.copy() for _ in range(n // base.number_of_nodes() + 1)])

        G = nx.convert_node_labels_to_integers(G)
        G = G.subgraph(range(n)).copy()

        if not nx.is_connected(G):
            largest_cc = max(nx.connected_components(G), key=len)
            G = G.subgraph(largest_cc).copy()

            # re-expand if needed
            while G.number_of_nodes() < n:
                u = random.choice(list(G.nodes()))
                new = G.number_of_nodes()
                G.add_node(new)
                G.add_edge(new, u)

        return G


    elif gtype == "LESMIS":

        base = nx.les_miserables_graph()

        G = nx.disjoint_union_all([base.copy() for _ in range(n // base.number_of_nodes() + 1)])

        G = nx.convert_node_labels_to_integers(G)
        G = G.subgraph(range(n)).copy()

        if not nx.is_connected(G):
            largest_cc = max(nx.connected_components(G), key=len)
            G = G.subgraph(largest_cc).copy()

            # re-expand if needed
            while G.number_of_nodes() < n:
                u = random.choice(list(G.nodes()))
                new = G.number_of_nodes()
                G.add_node(new)
                G.add_edge(new, u)
        return G

    
    else:
        raise ValueError("Unknown graph type")



        


# ================= FIRE SIMULATION =================
# ================= FIRE SIMULATION =================
def simulate_fire(G, protected, p, runs=20):

    nodes = list(G.nodes())
    candidates = [v for v in nodes if v not in protected]

    if len(candidates) == 0:
        return 0

    total_saved = 0

    for _ in range(runs):

        start = random.choice(candidates)

        burned = {start}
        frontier = [start]

        while frontier:

            new = []

            for u in frontier:
                for v in G.neighbors(u):

                    if v in protected or v in burned:
                        continue

                    if random.random() < p:
                        burned.add(v)
                        new.append(v)

            frontier = new

        saved = set(nodes) - burned - protected
        total_saved += len(saved)

    return total_saved / runs
# ================= COLLECTIVE INFLUENCE =================
def collective_influence(G, v, l=2):
    ball = nx.single_source_shortest_path_length(G, v, cutoff=l)
    boundary = [u for u,d in ball.items() if d == l]
    kv = G.degree(v)
    return (kv - 1) * sum(G.degree(u) - 1 for u in boundary)


# ================= SZEGEDY WALK =================
# (UNCHANGED)
def szegedy_quantum_metrics(G, steps):
    nodes = list(G.nodes())
    n = len(nodes)
    A = nx.to_numpy_array(G)
    deg = A.sum(axis=1)

    P = np.zeros_like(A)
    for i in range(n):
        if deg[i] > 0:
            P[i] = A[i] / deg[i]

    edges = [(i, j) for i in range(n) for j in range(n) if P[i, j] > 0]
    m = len(edges)
    edge_index = {e: k for k, e in enumerate(edges)}

    psi = np.zeros((n, m))
    for i, j in edges:
        psi[i, edge_index[(i, j)]] = np.sqrt(P[i, j])

    RA = 2 * psi.T @ psi - np.eye(m)

    S = np.zeros((m, m))
    for (i, j), k in edge_index.items():
        if (j, i) in edge_index:
            S[k, edge_index[(j, i)]] = 1

    RB = S @ RA @ S
    W = RB @ RA

    state = np.ones(m) / np.sqrt(m)
    for _ in range(steps):
        state = W @ state

    prob_e = np.abs(state) ** 2

    prob_v = np.zeros(n)
    for (i, j), k in edge_index.items():
        prob_v[i] += prob_e[k]
        prob_v[j] += prob_e[k]

    prob_v /= prob_v.sum()
    hit_time = 1 / (prob_v + 1e-12)

    eigvals = np.linalg.eigvalsh(A)
    lam = eigvals[-1]
    shift = np.zeros(n)

    for v in range(n):
        H = G.copy()
        H.remove_node(nodes[v])
        A2 = nx.to_numpy_array(H)
        if A2.size > 0:
            shift[v] = lam - np.linalg.eigvalsh(A2)[-1]

    return hit_time, shift



# ================= RANDOM WALK WITH RESTART =================
def classical_random_walk_scores(G, steps=100, alpha=0.15):

    nodes = list(G.nodes())
    n = len(nodes)

    A = nx.to_numpy_array(G)
    deg = A.sum(axis=1)

    P = np.zeros_like(A)

    for i in range(n):
        if deg[i] > 0:
            P[i] = A[i] / deg[i]

    # restart distribution (uniform)
    r = np.ones(n) / n

    state = r.copy()

    for _ in range(steps):
        state = (1 - alpha) * (state @ P) + alpha * r

    return state    


#===========================PSO=============================
# =========================================================
# PARTICLE SWARM OPTIMIZATION FOR FIREBREAK
# =========================================================

def pso_select(G, k, p_fire, runs=8):

    nodes = list(G.nodes())
    n = len(nodes)
    node_array = np.array(nodes)

    particles = pso_particles
    iters = pso_iters

    w_max = 0.9
    w_min = 0.4
    c1 = 1.7
    c2 = 1.7

    # --- particle position and velocity ---
    X = np.random.rand(particles, n)
    V = np.random.randn(particles, n) * 0.1

    # --- centrality seeding (VERY IMPORTANT) ---
    deg = dict(G.degree())
    deg_nodes = sorted(deg, key=deg.get, reverse=True)[:k]

    bc = nx.betweenness_centrality(G)
    bc_nodes = sorted(bc, key=bc.get, reverse=True)[:k]

    ci = {v: collective_influence(G, v, l_ci) for v in G.nodes()}
    ci_nodes = sorted(ci, key=ci.get, reverse=True)[:k]

    seed_sets = [deg_nodes, bc_nodes, ci_nodes]

    for s, nodeset in enumerate(seed_sets):

        if s >= particles:
            break

        X[s] = 0
        idx = [np.where(node_array == v)[0][0] for v in nodeset]
        X[s, idx] = 1

    # --- decode vector into node set ---
    def decode(x):
        idx = np.argpartition(-x, k)[:k]
        return set(node_array[idx])

    # --- initialize personal best ---
    pbest = X.copy()
    pbest_score = np.zeros(particles)

    for i in range(particles):
        S = decode(X[i])
        pbest_score[i] = simulate_fire(G, S, p_fire, runs)

    # --- global best ---
    gbest_idx = np.argmax(pbest_score)
    gbest = pbest[gbest_idx].copy()

    # --- optimization loop ---
    for t in range(iters):

        w = w_max - (w_max - w_min) * (t / iters)

        for i in range(particles):

            r1 = np.random.rand(n)
            r2 = np.random.rand(n)

            V[i] = (
                w * V[i]
                + c1 * r1 * (pbest[i] - X[i])
                + c2 * r2 * (gbest - X[i])
            )

            X[i] += V[i]
            X[i] = np.clip(X[i], 0, 1)

            S = decode(X[i])
            score = simulate_fire(G, S, p_fire, runs)

            if score > pbest_score[i]:
                pbest[i] = X[i].copy()
                pbest_score[i] = score

        gbest_idx = np.argmax(pbest_score)
        gbest = pbest[gbest_idx].copy()

    return decode(gbest)

    
#======================ACO==============================
# =========================================================
# ANT COLONY OPTIMIZATION FOR FIREBREAK
# =========================================================

def aco_select(G, k, p_fire, runs=8):

    nodes = list(G.nodes())
    n = len(nodes)

    ants = aco_ants
    iters = aco_iters

    pheromone = np.ones(n)

    node_index = {node: i for i, node in enumerate(nodes)}

    best_set = None
    best_score = -1

    # heuristic: degree importance
    deg = np.array([G.degree(v) for v in nodes]) + 1

    for _ in range(iters):

        iteration_best = None
        iteration_score = -1

        # --- probability distribution ---
        prob = pheromone * (deg ** beta)
        prob = prob / prob.sum()

        for _ in range(ants):

            idx = np.random.choice(np.arange(n), k, replace=False, p=prob)

            candidate = {nodes[i] for i in idx}

            score = simulate_fire(G, candidate, p_fire, runs)

            if score > iteration_score:
                iteration_score = score
                iteration_best = candidate

            if score > best_score:
                best_score = score
                best_set = candidate

        # --- pheromone evaporation ---
        pheromone *= (1 - rho)

        # --- pheromone reinforcement ---
        for node in iteration_best:
            pheromone[node_index[node]] += iteration_score / n

    return best_set

# # ================= MINIMUM BUDGET SOLVER =================
# def minimum_budget_required(G, method):

#     n = G.number_of_nodes()

#     if method == "BC":
#         score = nx.betweenness_centrality(G)

#     elif method == "CI":
#         score = {v: collective_influence(G,v,l_ci) for v in G.nodes()}

#     elif method in ["QH","SS"]:
#         hit_time, shift = szegedy_quantum_metrics(G, quantum_steps)
#         if method == "QH":
#             score = {i: -hit_time[i] for i in range(n)}
#         else:
#             score = {i: shift[i] for i in range(n)}

#     elif method == "DEG":
#         score = dict(G.degree())

#     # -------- METAHEURISTICS --------
#     elif method == "PSO":
#         # Incremental search over k
#         for k in range(1, n):
#             protected = pso_select(G, k)
#             saved = simulate_fire(G, protected, p_fire, mc_runs)

#             if saved >= target_save_fraction * n:
#                 return k, saved

#         return n, 0

    # elif method == "ACO":
    #     for k in range(1, n):
    #         protected = aco_select(G, k)
    #         saved = simulate_fire(G, protected, p_fire, mc_runs)

    #         if saved >= target_save_fraction * n:
    #             return k, saved

    #     return n, 0

   # else:
    #    raise ValueError("Unknown method")

    # -------- CENTRALITY METHODS --------
    ranked = sorted(score, key=score.get, reverse=True)

    for k in range(1, n):

        protected = set(ranked[:k])
        saved = simulate_fire(G, protected, p_fire, mc_runs)

        if saved >= target_save_fraction * n:
            return k, saved

    return n, 0    


# ================= RESULTS STORAGE =================
saved_bc, saved_qh, saved_ci, saved_ss, saved_deg, saved_rand, saved_crw = [], [], [], [], [], [], []
saved_pso, saved_aco = [], []
min_bc, min_ci, min_qh, min_ss, min_deg, min_pso, min_aco = [], [], [], [], [], [], []


# ================= MAIN LOOP =================
for n in sizes:

    G = make_graph(n, graph_type)
    k = max(1, int(budget_frac * n))

    bc = nx.betweenness_centrality(G)
    bc_nodes = sorted(bc, key=bc.get, reverse=True)[:k]

    ci = {v: collective_influence(G, v, l_ci) for v in G.nodes()}
    ci_nodes = sorted(ci, key=ci.get, reverse=True)[:k]

    hit_time, shift = szegedy_quantum_metrics(G, quantum_steps)
    qh_nodes = np.argsort(hit_time)[:k]
    ss_nodes = np.argsort(-shift)[:k]

    deg = dict(G.degree())
    deg_nodes = sorted(deg, key=deg.get, reverse=True)[:k]

        # --- Classical Random Walk ---
    crw = classical_random_walk_scores(G, 100)
    crw_nodes = np.argsort(-crw)[:k]

    # # --- NEW METHODS ---
    # --- metaheuristic search uses small simulation runs ---
    pso_nodes = pso_select(G, k, p_fire, runs=8)
    aco_nodes = aco_select(G, k, p_fire, runs=8)
    
    rand_total = 0
    for _ in range(20):
        rand_nodes = random.sample(list(G.nodes()), k)
        rand_total += simulate_fire(G, set(rand_nodes), p_fire, mc_runs)
    saved_rand.append(rand_total / 20)

    saved_bc.append(simulate_fire(G, set(bc_nodes), p_fire, mc_runs))
    saved_ci.append(simulate_fire(G, set(ci_nodes), p_fire, mc_runs))
    saved_qh.append(simulate_fire(G, set(qh_nodes), p_fire, mc_runs))
    saved_ss.append(simulate_fire(G, set(ss_nodes), p_fire, mc_runs))
    saved_deg.append(simulate_fire(G, set(deg_nodes), p_fire, mc_runs))
    saved_crw.append(simulate_fire(G, set(crw_nodes), p_fire, mc_runs))

    saved_pso.append(simulate_fire(G, pso_nodes, p_fire, mc_runs))
    saved_aco.append(simulate_fire(G, aco_nodes, p_fire, mc_runs))
    #     # ===== MINIMUM BUDGET COMPUTATION =====

    # k_bc, _  = minimum_budget_required(G, "BC")
    # k_ci, _  = minimum_budget_required(G, "CI")
    # k_qh, _  = minimum_budget_required(G, "QH")
    # k_ss, _  = minimum_budget_required(G, "SS")
    # k_deg, _ = minimum_budget_required(G, "DEG")
    # k_pso, _ = minimum_budget_required(G, "PSO")
    # k_aco, _ = minimum_budget_required(G, "ACO")

    # min_bc.append(k_bc)
    # min_ci.append(k_ci)
    # min_qh.append(k_qh)
    # min_ss.append(k_ss)
    # min_deg.append(k_deg)
    # min_pso.append(k_pso)
    # min_aco.append(k_aco)

# ================= GRAPH DRAWING FOR CURRENT CLASS (n=250) =================
n_draw = 250
G = make_graph(n_draw, graph_type)

plt.figure(figsize=(6,6))
pos = nx.spring_layout(G, seed=seed)
nx.draw(G, pos, node_size=25, edge_color="gray", with_labels=False)
plt.title(f"{graph_type} Graph (n=250)")
plt.savefig(f"{graph_type}.png", dpi=300, bbox_inches="tight")
plt.show()



# ================= PERFORMANCE PLOT =================
plt.figure(figsize=(8, 6))
plt.plot(sizes, saved_bc, 'o-', label="Betweenness")
plt.plot(sizes, saved_qh, 's-', label="Quantum")
plt.plot(sizes, saved_ci, '^-', label="CI")
plt.plot(sizes, saved_ss, 'd-', label="Spectral")
plt.plot(sizes, saved_deg, '*-', label="Degree")
plt.plot(sizes, saved_rand, 'x-', label="Random")
plt.plot(sizes, saved_crw, 'v-', label="Classical RW")
plt.plot(sizes, saved_pso, 'p-', label="PSO")
plt.plot(sizes, saved_aco, 'h-', label="ACO")
plt.xlabel("Number of Vertices")
plt.ylabel("TRUE Saved Vertices")
plt.title(f"Firebreak Performance ({graph_type})")
plt.legend()
plt.grid(True)
plt.show()


# ================= SAVED RATIO PLOT =================
ratio_bc = [saved_bc[i] / sizes[i] for i in range(len(sizes))]
ratio_qh = [saved_qh[i] / sizes[i] for i in range(len(sizes))]
ratio_ci = [saved_ci[i] / sizes[i] for i in range(len(sizes))]
ratio_ss = [saved_ss[i] / sizes[i] for i in range(len(sizes))]
ratio_deg = [saved_deg[i] / sizes[i] for i in range(len(sizes))]
ratio_rand = [saved_rand[i] / sizes[i] for i in range(len(sizes))]
ratio_crw = [saved_crw[i] / sizes[i] for i in range(len(sizes))]
ratio_pso = [saved_pso[i] / sizes[i] for i in range(len(sizes))]
ratio_aco = [saved_aco[i] / sizes[i] for i in range(len(sizes))]

plt.figure(figsize=(8,6))
plt.plot(sizes, ratio_bc, 'o-', label="Betweenness")
plt.plot(sizes, ratio_qh, 's-', label="Quantum")
plt.plot(sizes, ratio_ci, '^-', label="CI")
plt.plot(sizes, ratio_ss, 'd-', label="Spectral")
plt.plot(sizes, ratio_deg, '*-', label="Degree")
plt.plot(sizes, ratio_rand, 'x-', label="Random")
plt.plot(sizes, ratio_crw, 'v-', label="Classical RW")
plt.plot(sizes, ratio_pso, 'p-', label="PSO")
plt.plot(sizes, ratio_aco, 'h-', label="ACO")
plt.ylim(0,1.05)
plt.legend()
plt.grid(True)
plt.show()


# ================= RESULTS TABLE =================
print("\nResults Table (TRUE Saved Vertices — Protected Excluded)\n")
header = f"{'n':>6} | {'Betweenness':>12} | {'Quantum':>10} | {'CI':>10} | {'Spectral':>10} | {'Degree':>10} | {'CRW':>10} | {'Random':>10} | {'PSO':>8} | {'ACO':>8}"
#header = f"{'n':>6} | {'Betweenness':>12} | {'Quantum':>10} | {'CI':>10} | {'Spectral':>10} | {'Degree':>10} | {'Random':>10} | {'PSO':>8} | {'ACO':>8}"
print(header)
print("-"*len(header))

for i,n in enumerate(sizes):
    print(f"{n:6d} | {saved_bc[i]:12.2f} | {saved_qh[i]:10.2f} | {saved_ci[i]:10.2f} | {saved_ss[i]:10.2f} | {saved_deg[i]:10.2f} | {saved_crw[i]:10.2f} | {saved_rand[i]:10.2f} | {saved_pso[i]:8.2f} | {saved_aco[i]:8.2f}")
    #print(f"{n:6d} | {saved_bc[i]:12.2f} | {saved_qh[i]:10.2f} | {saved_ci[i]:10.2f} | {saved_ss[i]:10.2f} | {saved_deg[i]:10.2f} | {saved_rand[i]:10.2f}")
    #print(f"{n:6d} | {saved_bc[i]:12.2f} | {saved_qh[i]:10.2f} | {saved_ci[i]:10.2f} | {saved_ss[i]:10.2f} | {saved_deg[i]:10.2f} | {saved_rand[i]:10.2f} | {saved_pso[i]:8.2f} | {saved_aco[i]:8.2f}")
#     # ================= MINIMUM BUDGET PLOT =================
# plt.figure(figsize=(8,6))

# plt.plot(sizes, min_bc, 'o-', label="Betweenness")
# plt.plot(sizes, min_qh, 's-', label="Quantum Hitting")
# plt.plot(sizes, min_ci, '^-', label="CI")
# plt.plot(sizes, min_ss, 'd-', label="Spectral")
# plt.plot(sizes, min_deg, '*-', label="Degree")
# plt.plot(sizes, min_pso, 'p-', label="PSO")
# plt.plot(sizes, min_aco, 'h-', label="ACO")

# plt.xlabel("Number of Vertices")
# plt.ylabel("Minimum Budget Required")
# plt.title(f"Minimum Budget to Save {int(target_save_fraction*100)}% ({graph_type})")
# plt.legend()
# plt.grid(True)
# plt.show()


# print("\nMinimum Budget Required to Save "
#       f"{int(target_save_fraction*100)}% of Vertices\n")

# header = f"{'n':>6} | {'BC':>6} | {'QH':>6} | {'CI':>6} | {'SS':>6} | {'DEG':>6} | {'PSO':>6} | {'ACO':>6}"
# print(header)
# print("-"*len(header))

# for i,n in enumerate(sizes):
#     print(f"{n:6d} | {min_bc[i]:6d} | {min_qh[i]:6d} | {min_ci[i]:6d} | {min_ss[i]:6d} | {min_deg[i]:6d} | {min_pso[i]:6d} | {min_aco[i]:6d}")

KeyboardInterrupt: 